# 23-19 · Читаем и проверяем TOML-настройки

Практика к разделу [«Настройки проекта»](../../site/chapters/glava-23/23-22-nastrojki-proekta.html). Настоящий файл — `projects/python/safesort/src/safesort/config.py`.

## Цель

Разобрать `safesort.toml`-подобный текст через `tomllib` и построить из него `Config`-подобный объект — с теми же значениями по умолчанию, что и настоящий `load_config()`.

## Рабочий пример

In [1]:
import tomllib
from dataclasses import dataclass, field

DEFAULT_DESTINATION = "Sorted"
DEFAULT_EXCLUDE = (".git", ".venv")
DEFAULT_EXTENSIONS = {
    "documents": [".pdf", ".docx", ".txt", ".odt"],
    "images": [".jpg", ".jpeg", ".png", ".webp"],
}


@dataclass(frozen=True)
class Config:
    destination: str = DEFAULT_DESTINATION
    exclude: tuple = field(default_factory=lambda: DEFAULT_EXCLUDE)
    extensions: dict = field(default_factory=lambda: {k: list(v) for k, v in DEFAULT_EXTENSIONS.items()})


TEKST_TOML = """
destination = "Archive"
exclude = [".git", ".venv", "node_modules"]

[extensions]
documents = [".pdf", ".docx", ".txt"]
images = [".jpg", ".jpeg", ".png", ".webp"]
"""

raw = tomllib.loads(TEKST_TOML)
print(raw)

{'destination': 'Archive', 'exclude': ['.git', '.venv', 'node_modules'], 'extensions': {'documents': ['.pdf', '.docx', '.txt'], 'images': ['.jpg', '.jpeg', '.png', '.webp']}}


## Проверка результата — TOML разобран в обычный словарь Python

In [2]:
assert raw["destination"] == "Archive"
assert raw["exclude"] == [".git", ".venv", "node_modules"]
assert raw["extensions"]["documents"] == [".pdf", ".docx", ".txt"]
print("Верно: tomllib.loads() вернул обычный словарь с ожидаемой структурой.")

Верно: tomllib.loads() вернул обычный словарь с ожидаемой структурой.


## Строим Config из разобранного TOML

In [3]:
def config_from_raw(raw):
    return Config(
        destination=raw.get("destination", DEFAULT_DESTINATION),
        exclude=tuple(raw.get("exclude", list(DEFAULT_EXCLUDE))),
        extensions=raw.get("extensions", {k: list(v) for k, v in DEFAULT_EXTENSIONS.items()}),
    )


nastrojki = config_from_raw(raw)
print(nastrojki)

Config(destination='Archive', exclude=('.git', '.venv', 'node_modules'), extensions={'documents': ['.pdf', '.docx', '.txt'], 'images': ['.jpg', '.jpeg', '.png', '.webp']})


## Задание ★ Базовая практика

Разберите TOML без секции `[extensions]` и убедитесь, что `config_from_raw()` подставляет `DEFAULT_EXTENSIONS`, — точно так же, как это делает настоящий `load_config()` при отсутствующем ключе.

In [4]:
TEKST_BEZ_EXTENSIONS = 'destination = "Sorted2"\n'
raw_minimalnyj = tomllib.loads(TEKST_BEZ_EXTENSIONS)
nastrojki_minimalnye = config_from_raw(raw_minimalnyj)

assert nastrojki_minimalnye.destination == "Sorted2"
assert nastrojki_minimalnye.extensions == DEFAULT_EXTENSIONS
assert nastrojki_minimalnye.exclude == DEFAULT_EXCLUDE
print("Верно: без явных настроек используются значения по умолчанию.")

Верно: без явных настроек используются значения по умолчанию.
